In [ ]:
from glob import glob
import pandas as pd
from tqdm import tqdm

from utils import extract_url_map
from events import extract_events
from timespan import parse_timespan
from taxonomy import load_taxonomy

In [ ]:
# root_path = "../data/schede mappatura/"
root_path = "../data/nuove schede/"

skip = {
    # "david_ruth_FEGB_E_00007": {"rows": 6, "cols": 1}
    "stern_IS_S_00142": {"rows": 2, "cols": 0},
    "chronotopoi_josef_stern_familie_v4a.xlsx": {"rows": 2, "cols": 0},
}

In [ ]:
chrono_schede = glob(f"{root_path}*chronotop*")
chrono_schede = [c for c in chrono_schede if not c.endswith(":Zone.Identifier")]
chrono_schede

## A list of individual sources for experimentation

ignored in the oveall logic

In [ ]:
# current = "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx"
# current = "david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx"
# current = "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx"
# current = "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx"
current = "chronotopoi_josef_stern_familie_v4a.xlsx"

chrono_schede = [s for s in chrono_schede if current in s]
chrono_schede

After identification of all sources

# Shortlist processable sources

In [ ]:
overview = {
    "chronotopoi_josef_stern_familie_v4a.xlsx": [
        "Josef Stern",
    ],
}

## Put all data together

In [ ]:
first = True
df_list = []
for k, v in overview.items():
    for s in v:
        # Extract name from k: try both nested path (for schede mappatura) and filename (for nuove schede)
        name = k.split("/")[0] if "/" in k else k
        # print(k,s)
        if name in skip:
            df = pd.read_excel(
                root_path + k, sheet_name=s, skiprows=skip[name]["rows"]
            ).iloc[:, skip[name]["cols"] :]
        else:
            df = pd.read_excel(root_path + k, sheet_name=s)

        print(k, s, f"Columns: {list(df.columns)}")
        
        # The Excel file already has proper column names. Just ensure we have the ones we need.
        # If columns are named correctly, we can use them directly.
        # Otherwise, rename to the expected schema.
        
        # Only rename if the first column is not a proper name (i.e., when we have Unnamed columns)
        if "Unnamed:" in str(df.columns[0]):
            # Old format - needs renaming
            df.columns = [
                "event_label",
                "event_type",
                "place_name",
                "place_type",
                "place_category",
                "wikidata_qid",
                "geonames_id",
                "google maps",
                "date_certainty",
                "date_label",
                "memorial_inscription",
                "source_doc",
                "source_timecode",
                "source_quote",
                "external_links",
                "notes",
            ]
            # merge columns 6+ to notes
            df["notes"] = df[["notes"] + list(df.columns[6:])].apply(
                lambda row: " ".join(row.dropna().astype(str)), axis=1
            )
        else:
            # New format with proper column names - use as-is
            pass

        df["protagonist"] = name
        df["name"] = s
        df_list += [df]
df = pd.concat(df_list, axis=0).astype(str)
df.fillna("", inplace=True)

# Track start/end locations: end_location = current row's place,
# start_location = previous event's place (per person)
start_locations = []
prev_location = {}  # protagonist → last place_name
for _, row in df.iterrows():
    person = row["protagonist"]
    end_loc = row["place_name"].strip()
    start_loc = prev_location.get(person, end_loc)  # fallback to same as end
    start_locations.append(start_loc)
    if end_loc:
        prev_location[person] = end_loc
df["start_location"] = start_locations
df["end_location"] = df["place_name"]

# Collect concepts from event_type, place_type, place_category
concept_labels = set()
for col in ["event_type", "place_category"]:  # Adjusted for new columns
    if col in df.columns:
        for val in df[col]:
            v = val.strip() if isinstance(val, str) else str(val).strip()
            if v and v not in ("nan", "None", ""):
                concept_labels.add(v)
print(f"Concepts to create: {sorted(concept_labels)}")

df


# Locations

In [ ]:
# locs = {n:l for l, n in df["location"].apply(lambda x: extract_urls(x)).to_list()}
locs = {}
for row in tqdm(df.to_dict(orient="records")):
    # print(row)
    # print(extract_urls(row))
    name = row["place_name"]
    locs[name] = {}
    place_labels = set()
    if "place_type" in row and row["place_type"].strip():
        place_labels |= {row["place_type"].strip()}
    if row["place_category"].strip():
        place_labels |= {row["place_category"].strip()}
    locs[name]["label"] = ",".join(place_labels)

    urls = extract_url_map(row["external_links"])
    if (
        "www.wikidata.org" not in urls
        and "wikidata_id" in row
        and row["wikidata_qid"].strip()
    ):
        locs[name]["www.wikidata.org"] = (
            "https://www.wikidata.org/wiki/" + row["wikidata_qid"].strip()
        )
    if (
        "www.geonames.org" not in urls
        and "geonames_id" in row
        and row["geonames_id"].strip()
    ):
        locs[name]["www.geonames.org"] = (
            "https://www.geonames.org/" + row["geonames_id"].strip().removesuffix(".0")
        )

    locs[name].update(urls[0])
print(locs)

## Add GO concepts as locations

Concepts under 'GO – Luoghi geografici' are geographic locations.
Add them to the locs dict so they get included in locations.xlsx.

In [ ]:
from taxonomy import load_taxonomy

taxonomy = load_taxonomy()

go_concepts = [
    label for label, cat in taxonomy.concept_to_category.items()
    if cat == "GO"
]
# Also include GO sub-category labels
for key, info in taxonomy.sub_categories.items():
    if info.get("parent") == "GO":
        go_concepts.append(info["label"])

for concept_name in go_concepts:
    concept_name = concept_name.strip()
    if not concept_name:
        continue
    if concept_name not in locs:
        locs[concept_name] = {}
    if "label" not in locs[concept_name] or not locs[concept_name]["label"]:
        locs[concept_name]["label"] = "GO"

print(f"Added {len(go_concepts)} GO concepts to locations, total: {len(locs)}")


## Update preexisting locations

In [ ]:
import os
import re
from locations import enrich_locations_xlsx

def _normalize_loc(name):
    """Normalize for matching: lowercase, no punctuation, sorted words."""
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

if os.path.exists("locations.xlsx"):
    rich = pd.read_excel("locations.xlsx", dtype=str)
    rich = rich.set_index(["location"])
else:
    rich = pd.DataFrame()
    rich.index.name = "location"

# Build a normalized index for flexible matching
existing_keys = {_normalize_loc(idx): idx for idx in rich.index}

for location, row in locs.items():
    key = _normalize_loc(location)

    if key in existing_keys:
        # Enrich existing row: only fill absent cells
        real_idx = existing_keys[key]
        if isinstance(row, dict):
            for col, val in row.items():
                if col not in rich.columns:
                    rich[col] = ""
                existing = rich.loc[real_idx, col]
                if isinstance(existing, pd.Series):
                    existing = existing.iloc[0]
                if pd.isna(existing) or str(existing).strip() in ("", "nan"):
                    rich.loc[real_idx, col] = str(val)
    else:
        # New location: add row with provided values
        if isinstance(row, dict):
            for col in row:
                if col not in rich.columns:
                    rich[col] = ""
            rich.loc[location] = {col: str(val) for col, val in row.items()}
        else:
            rich.loc[location] = pd.Series(dtype=str)
        existing_keys[key] = location

rich.to_excel("locations.xlsx")

# Enrich with bag-of-words and super-region columns
enrich_locations_xlsx("locations.xlsx")

In [ ]:
taxonomy = load_taxonomy("../data/maxqda/MAXQDA_Code_System.mmd")
events = sorted(set(taxonomy.concept_to_category.keys()), key=lambda x: -len(x))
len(events), events[:15] + ["..."] + events[-15:]
# ",".join(events)

# Timespan

In [ ]:
df[["time_start", "time_end"]] = (
    df["date_label"].apply(lambda ts: list(parse_timespan(ts).as_tuple())).tolist()
)
df

# Notes

left unprocessed for now

In [ ]:
set(df["notes"])

# Links

left unprocessed for now

In [ ]:
urls = set()
for cell in df["external_links"]:
    if pd.notna(cell):
        for url in str(cell).split("\n"):
            url = url.strip()
            if url:
                urls.add(url)
urls

# Events

In [ ]:
df["event"] = df["event_label"].apply(lambda x: extract_events(x, events))
df

In [ ]:
from api_client import login, get_or_create_concept
from locations import load_locations_db

login()
load_locations_db()

print("Creating concepts...")
for label in sorted(concept_labels):
    get_or_create_concept(label)
print(f"  {len(concept_labels)} concept labels processed")

# Events

Persist chronotopoi events from DataFrame to the API with locations, dates, and categories.


In [ ]:
from api_client import api_post, api_patch
from locations import get_or_create_location, load_locations_db
from timespan import parse_date_range
from utils import is_empty

# Load locations database for coordinate lookup
locations_db = load_locations_db()

def create_event_from_row(row, protagonist, locations_db=None):
    """Create an Event object and persist to API from a chronotopi dataframe row.
    
    Args:
        row: Dict containing event data with keys: event_label, place_name, place_detail,
             event_type, transport_mode, place_category, date_start, date_end, external_links
        protagonist: Person identifier (name) for this event
        locations_db: Pre-loaded locations database for coordinate lookup
    
    Returns:
        Event ID if created successfully, None otherwise
    """
    from utils import extract_url_map
    
    # Build unique archive_id from protagonist + event_label + row index
    event_label = row.get("event_label", "").strip()
    if not event_label:
        return None
    
    # Create location from place_name + place_detail
    place_name = row.get("place_name", "").strip()
    place_detail = row.get("place_detail", "").strip()
    
    if not place_name:
        return None
    
    # Combine place_name and place_detail if both present
    location_name = place_name
    if place_detail:
        location_name = f"{place_name} ({place_detail})"
    
    # Get or create location
    location_id = get_or_create_location(
        location_name,
        wikidata_qid=row.get("wikidata_qid"),
        geonames_id=str(row.get("geonames_id", "")).rstrip(".0") if row.get("geonames_id") else None,
        locations_db=locations_db
    )
    
    # Parse dates
    date_start = row.get("date_start")
    date_end = row.get("date_end")
    start_iso, end_iso = parse_date_range(f"{date_start}" if date_start else "-")
    
    # Collect categories from event_type, transport_mode, place_category
    categories = []
    for field in ["event_type", "transport_mode", "place_category"]:
        val = row.get(field, "").strip()
        if val and val not in ("nan", ""):
            categories.append(val)
    
    # Extract URLs from external_links
    external_links = row.get("external_links", "")
    urls = extract_url_map(external_links) if external_links else []
    
    # Build event payload
    archive_id = f"{protagonist}_{event_label.replace(' ', '_')[:20]}"
    
    payload = {
        "archive_id": archive_id[:100],  # Limit to field max length
        "description": event_label,
    }
    
    # Add URLs if present
    if urls:
        url_ids = []
        for url_entry in urls:
            if isinstance(url_entry, dict) and "url" in url_entry:
                try:
                    from api_client import api_post as api_post_direct
                    url_obj = api_post_direct("urls", {"url": url_entry["url"]})
                    url_ids.append(url_obj["id"])
                except:
                    pass
        if url_ids:
            payload["urls"] = url_ids
    
    # Create event via API
    try:
        event = api_post("events", payload)
        event_id = event.get("id")
        
        if event_id and location_id:
            # Link location to event (if Event model has a location field)
            try:
                api_patch(f"events/{event_id}", {"location": location_id})
            except:
                pass
        
        return event_id
    except Exception as e:
        print(f"  Error creating event {archive_id}: {e}")
        return None

# Create events from all dataframe rows
print("Creating events from chronotopi data...")
created_events = []
for idx, row in df.iterrows():
    protagonist = row.get("protagonist", "Unknown")
    event_id = create_event_from_row(row, protagonist, locations_db)
    if event_id:
        created_events.append(event_id)
        print(f"  ✅ {row['event_label']} → Event #{event_id}")
    else:
        print(f"  ⚠️ Skipped: {row['event_label']}")

print(f"\n✅ Created {len(created_events)} events")


In [ ]:
# Link concepts and locations to events

from api_client import api_get

# Get all created events for reference
all_events = api_get("events")
event_map = {evt.get("archive_id"): evt for evt in all_events if evt.get("archive_id")}

# Get all concepts for linking
all_concepts = api_get("concepts")
concept_map = {
    con.get("label", "").lower().replace(" ", "_"): con["id"]
    for con in all_concepts if con.get("label")
}

print(f"Linking {len(created_events)} events with concepts and locations...")

linked_count = 0
for idx, row in df.iterrows():
    protagonist = row.get("protagonist", "Unknown")
    event_label = row.get("event_label", "").strip()
    
    if not event_label:
        continue
    
    # Find the event we just created
    archive_id = f"{protagonist}_{event_label.replace(' ', '_')[:20]}"
    event = event_map.get(archive_id)
    
    if not event:
        continue
    
    event_id = event["id"]
    
    # Collect and link categories
    categories = []
    for field in ["event_type", "transport_mode", "place_category"]:
        val = row.get(field, "").strip()
        if val and val not in ("nan", ""):
            categories.append(val)
    
    # Link each category as a concept
    for category in categories:
        category_key = category.lower().replace(" ", "_")
        concept_id = concept_map.get(category_key)
        if concept_id:
            # Link concept to event (if relationship exists in API)
            try:
                # Try to link via extraction or event-concept relationship
                api_patch(f"events/{event_id}", {"concepts": [concept_id]})
                linked_count += 1
            except:
                pass  # Silently ignore if relationship doesn't exist

print(f"✅ Linked {linked_count} concept relationships")
print(f"\n📍 Events created and ready for map visualization!")
print(f"   - {len(created_events)} chronotopi events")
print(f"   - {len(set(df['place_name'].dropna()))} locations")
print(f"   - {len(concept_labels)} event concepts")
